In [1]:
import numpy as np
import pandas as pd 

from astroquery.sdss import SDSS 
from astropy.table import Table
from astropy.io import fits
import numpy as np
from tqdm import tqdm
import time

import os
import random

In [3]:
from requests.exceptions import ConnectionError, Timeout, ReadTimeout
from http.client import RemoteDisconnected
from urllib3.exceptions import ProtocolError
import socket

In [3]:
SDSS.TIMEOUT = 300

### Fetching the spectral data

In [4]:
class DataFetcher:

    '''
    Fetch stellar spectral data using SQL queries from SDSS sql sky server \n
    Object Attributes : \n
    spectral_class : string - from list ["O","B","F","K","M","A","G"] \n
    output_directory : string - saving directory
    '''

    def __init__(self, spectral_class, output_directory = "/kaggle/working"):
        self.spectral_class = spectral_class
        self.output_directory = output_directory

    def get_table(self):

        '''
        Request Query based on the spectral class
        '''

        if self.spectral_class == "F":
            num = 4290
        else:
            num = 4285

        # running the sql query to get the fiberid, mjd, plate for the stars
        sql_query = f'''SELECT TOP {num}
            specobjid, ra, dec,plate, mjd, fiberid, subclass, z, elodieTeff, elodieLogG, elodieFeH
            FROM SpecObj
            WHERE class = 'STAR' AND zWarning = 0 
              AND subclass LIKE '{self.spectral_class}%'
              AND (snMedian_u + snMedian_g + snMedian_r + snMedian_i + snMedian_z)/5.0 > 10
              AND elodieTeff IS NOT NULL AND elodieLogG IS NOT NULL AND elodieFeH IS NOT NULL'''

        res = SDSS.query_sql(sql_query, timeout = 600)

        return res

    # Defining a class method
    @classmethod
    def retrieve_spectra(cls, output_dir, plate_id, mjd, fiber_id, failed_ids, chunk_num, n, spectral_class):
        ''''
        Instant saving stellar spectra on successful https request 
        '''
        save_dir = os.path.join(output_dir, f"flux_array_{spectral_class}", "spectrum_fits", f"Chunk_{chunk_num}")
        os.makedirs(save_dir, exist_ok=True)    # defining the saving directory 
        
        spectra = None   # initialize the spectra as none 
        
        # attempting to get the spectra 5 times if unsuccessful 
        for attempt in range(5):
            try:
                SDSS.TIMEOUT = 300  # Increase timeout to 5 min
                
                # creating the http request 
                spectra = SDSS.get_spectra(
                    plate=plate_id, mjd=mjd, fiberID=fiber_id, data_release=19
                        )  
                if spectra is not None:
                    
                    # accessing the FITS format to retrieve the spectra
                    data_file = spectra[0]     
                    primary_hdu = data_file[0]
                    secondary_hdu = data_file[1]
    
                    header = primary_hdu.header
    
                    try:
                        obj_id = header["THING_ID"]    # unique object identifier
                    except:
                        obj_id = header["spec_id"]
    
                    secondary_data = secondary_hdu.data
                    flux = [row[0] for row in secondary_data]    # adding the unique object identifier on the first index of each star spectra
                    flux[0] = obj_id
                    
                    #save block
                    save_path = os.path.join(save_dir, f"spectrum_{n}.npy")
                    np.save(save_path, flux)
                    
                    break  # success
            
            # creating exception for all the timeout and connection errors
            except (ConnectionError, Timeout, ReadTimeout, socket.timeout, 
                                RemoteDisconnected, ProtocolError, TimeoutError) as e:
                
                wait_time = 10 * (2 ** attempt) + random.uniform(0, 5)     # creating randomized wait times for each exception
                print(f"{type(e).__name__}: Retry {attempt+1}/5 after {wait_time:.1f}s "
                      f"for plate={plate_id}, mjd={mjd}, fiber={fiber_id}")
                time.sleep(wait_time)
    
        if spectra is None:
            failed_ids.append((plate_id, mjd, fiber_id))   # appending failed spectra 
        
    # chunk wise loading spectra
    def chunk_wise_loader(self, chunk_size,output_directory):
        ''' 
        Chunk wise loading the spectra for the given spectral class\n
        Parameters : \n
        chunk_size : int \n
        output_directory : string \n
        '''
        res = self.get_table()
        print(len(res))

        flux_array = []
        spectral_class = self.spectral_class
        
        n = len(res)//chunk_size   # n chunks
        count = 0
        for i in tqdm(range(n)):
            try:
                new_res = res[chunk_size * i : chunk_size * (i + 1)]
            except:
                new_res = res[chunk_size * i :]     # last chunk exception
            spectra_list = []
            failed_ids = []
            
            for j in range(chunk_size):  # loading inside a chunk
                plate_id, mjd, fiber_id = new_res[j]["plate"], new_res[j]["mjd"], new_res[j]["fiberid"]
                
                DataFetcher.retrieve_spectra(output_directory, plate_id, mjd, fiber_id, failed_ids,chunk_num = i, n = j, spectral_class = spectral_class)

                # checker block

    def consolidate_data(self):
        '''
        Consolidating all the npy spectra flux arrays
        '''
        input_directory = "/kaggle/input/sdss-dissertation-data-1"
        directory_path = f"{input_directory}/flux_array_{self.spectral_class}/spectrum_fits"
        
        chunk_list_dir = os.listdir(directory_path)
        spectra_list = []
        
        for chunk_dir in tqdm(chunk_list_dir):
            chunk_spectra_list_dir = os.listdir(f"{directory_path}/{chunk_dir}")
        
            for j in chunk_spectra_list_dir:
                spectra = np.load(f"{directory_path}/{chunk_dir}/{j}")
                spectra_list.append(spectra)

        np.save(f"{self.output_directory}/spectra_{self.spectral_class}", np.array(spectra_list, dtype = object))


In [ ]:
spectral_class_list = ["A","F","G","K","M","O","B"]

for i in spectral_class_list:
    spectra_object= DataFetcher(spectral_class = i)
    spectra_object.chunk_wise_loader(chunk_size = 200 if spectra_object.spectral_class in ["O", "B"] else 500, output_directory = "/kaggle/working")

In [ ]:
spectra_class_list = ["O","B","A","F","G","K","M"]     

for i in spectra_class_list:
    spectra_object = DataFetcher(spectral_class = i)
    spectra_object.consolidate_data()

ModuleNotFoundError: No module named 'astroquery.lamost'